# Music Player Application with Kafka and Spark Streaming

## Overview
In this assignment, you will develop a music player application that streams user interaction events (like song plays, likes, and dislikes) to Apache Kafka. These events will be processed in real-time using Spark Structured Streaming to generate music recommendations based on user preferences, song metadata, and business goals to maximize revenue.

## Objectives
- Use the existing music player interface made Streamlit that randomly selects tracks.
- Stream user events to a Kafka topic.
- Process and analyze streaming data with Spark Structured Streaming.
- Implement a recommendation system based on user interactions and song metadata.
- Ensure recommendations are triggered after at least 10 user actions.

## Teams
You may work in groups of two or three. 

## Requirements

### Part 1: Music Player Interface (Streamlit)
- Modify the provided Streamlit application to include features for playing music, liking, disliking, and skipping tracks.
- For each user action, generate an event containing:
  - User ID
  - Track ID
  - Action type (Play, Like, Dislike,..)
  - Timestamp
  - If you want, you can also stream additional metadata (e.g., track length, album, artist, genre)
- Stream these events to a custom kafka topic of your choice. 
- ONE Topic per Group.

### Part 2: Real-Time Data Processing with Spark, Faust, or native Kafka API
- Develop a Spark Structured Streaming application to consume events from the Kafka topic.
- (You can also process the events using Python Kafka Api or Faust )
- Process and analyze the streaming data to track user preferences and song popularity.
- Implement logic to trigger song recommendations based on:
  - At least 10 user interactions.
  - Preferences for artists, genres, and albums,....
  - Maximization of revenue (considering factors like track pricing).

### Part 3: Generating Recommendations
- Use the processed data to generate song recommendations.
- Recommendations should be dynamic, reflecting the user's recent actions and general preferences.
- Display these recommendations in the Streamlit app, however you prefer. 
- Custom table from a database is also fine.

# Solution

**Group**: Lara Schwabe, Anna Tschon, Mered Wolde

**Topic Name**: endlich_ferien

In [1]:
topic_name = "endlich_ferien"

## Setup Kafka Topic

In [2]:
import utils.ccloud_lib as ccloud_lib
kafka_app_config = ccloud_lib.read_ccloud_config("kafka.config")

In [3]:
### Create a topic
from confluent_kafka.admin import AdminClient, NewTopic

admin_client = AdminClient(ccloud_lib.pop_schema_registry_params_from_config(kafka_app_config.copy()))
# For a single-broker Redpanda setup we use replication_factor=1.
# Increase it only when your cluster has enough brokers to host the replicas.
futures_result = admin_client.create_topics([NewTopic(topic_name, num_partitions=1, replication_factor=1)])
display(futures_result)

%4|1783269253.915|CONFWARN|fhtw-kafka-client#producer-1| [thrd:app]: Configuration property session.timeout.ms is a consumer property and will be ignored by this producer instance
%4|1783269253.915|CONFWARN|fhtw-kafka-client#producer-1| [thrd:app]: Configuration property heartbeat.interval.ms is a consumer property and will be ignored by this producer instance


{'endlich_ferien': <Future at 0x7f6628065010 state=running>}

In [4]:
# Wait and check for the creation of the topic
for topic, f in futures_result.items():
    f.result()  # The result itself is None
    print("Topic {} created".format(topic))

KafkaException: KafkaError{code=TOPIC_ALREADY_EXISTS,val=36,str="The topic has already been created"}

## Consume messages to check Streamlit functionality

In [5]:
group_id = "ferien_consumer"
kafka_app_config["group.id"] = group_id

In [6]:
kafka_app_config["auto.offset.reset"] = "earliest"

In [7]:
from confluent_kafka import Consumer

# create a consumer object
consumer = Consumer(kafka_app_config)
consumer.subscribe([topic_name])

%4|1783269261.125|CONFWARN|fhtw-kafka-client#consumer-2| [thrd:app]: Configuration property request.required.acks is a producer property and will be ignored by this consumer instance


In [8]:
# Processing Kafka Messages with a Consumer

import json
def run_consumer(consumer):
    while True:
        msg = consumer.poll(0.5)
        if msg is None:
            # No message available within timeout.
            # Initial message consumption may take up to
            # `session.timeout.ms` for the consumer group to
            # rebalance and start consuming
            print("Waiting for message or event/error in poll()", end="\r", flush=True)
            continue
        elif msg.error():
            print('error: {}'.format(msg.error()))
        else:
            # Check for Kafka message
            record_key = msg.key()
            record_value = msg.value()
            data = json.loads(record_value)
            print(data)

In [9]:
run_consumer(consumer)

{'user_id': 'user_ea3ade8c', 'track_id': 2493, 'action_type': 'Play', 'timestamp': '2026-07-05T16:25:21.263036+00:00', 'track_name': 'Disarm', 'artist_id': 131, 'artist_name': 'Smashing Pumpkins', 'album_id': 202, 'album_name': 'Rotten Apples: Greatest Hits', 'track_length_ms': 198556, 'unit_price': 0.99}
{'user_id': 'user_ea3ade8c', 'track_id': 2493, 'action_type': 'Like', 'timestamp': '2026-07-05T16:25:29.732284+00:00', 'track_name': 'Disarm', 'artist_id': 131, 'artist_name': 'Smashing Pumpkins', 'album_id': 202, 'album_name': 'Rotten Apples: Greatest Hits', 'track_length_ms': 198556, 'unit_price': 0.99}
{'user_id': 'user_ea3ade8c', 'track_id': 2493, 'action_type': 'Play', 'timestamp': '2026-07-05T16:25:32.248058+00:00', 'track_name': 'Disarm', 'artist_id': 131, 'artist_name': 'Smashing Pumpkins', 'album_id': 202, 'album_name': 'Rotten Apples: Greatest Hits', 'track_length_ms': 198556, 'unit_price': 0.99}
{'user_id': 'user_ea3ade8c', 'track_id': 2493, 'action_type': 'Dislike', 'times

KeyboardInterrupt: 

In [10]:
consumer.close()